# Tutorial 31 — Optimal-Design DOE (Sobol+Refine)

Companion to **§33.5** of the GMSH User Guide. Mirrors the FinStructBench case study from Jin & Sudjianto (JASA 2025) at tutorial scale.

Why bother with optimal design? A standard benchmark measures average accuracy on a fixed test set. Under the same API budget, the harness can do something stronger — identify *which factors* of the evaluation context drive failures (clarity, noise, role, instruction conflict, paraphrase depth, ...). That requires informative design points, not just numerous ones.

This notebook walks through:

1. **Side-by-side design quality**: `method="sobol"` vs `method="sobol+refine"` on the same factor catalog.
2. **Plug the design into the harness**: `DOEGMSBenchmark` honors `config.method` directly.
3. **Factor-effect analysis**: `DOEAnalyzer` logistic regression. The headline gain is in joint-model fit (pseudo-R², AIC), not necessarily individual stars.
4. **Scaling to paper settings** (n = 880, comprehensive preset).

In [ ]:
# === Preamble (run from kg-memory root) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd

from knowlytix.harness.graphdoe.design_matrix import DesignMatrix
from knowlytix.harness.graphdoe.factor_loader import FactorLoader
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Build matched designs and compare quality

We use the `comprehensive` preset (20 contextual factors — this is the same set the paper experiments on) at tutorial scale `n=128`. Replace with `n=880` for paper-scale results.

The four cells below are pure design generation — no LLM calls, runs in seconds on GPU and a minute or two on CPU.

In [ ]:
FACTOR_GROUP = "comprehensive"
N_RUNS = 128
SEED = 42

factors = FactorLoader().load_factors({"use_group": FACTOR_GROUP})
print(f"Factors: {len(factors)}")
for f in factors[:5]:
    cats = f.get("categories", [])
    print(f"  {f['name']:24s}  |L|={len(cats):2d}  e.g. {cats[:3]}")
print("  ...")

In [ ]:
# Sobol baseline
dm_sobol = DesignMatrix(factors, method="sobol", n_runs=N_RUNS, seed=SEED)
dm_sobol.generate()
q_sobol = dm_sobol.quality()
print(f"Sobol: {q_sobol}")

In [ ]:
# Sobol+Refine — GPU phi_p optimization on top of the same Sobol init
dm_refine = DesignMatrix(factors, method="sobol+refine", n_runs=N_RUNS, seed=SEED, verbose=True)
dm_refine.generate()
q_refine = dm_refine.quality()
print(f"Sobol+Refine: {q_refine}")
print(f"Refinement: {dm_refine._refine_info}")

In [ ]:
# Side-by-side comparison
rows = []
for k in ("phi_p", "min_dist", "mean_nn", "mir_avg", "mir_max", "coverage", "empty_cells"):
    rows.append({
        "metric": k,
        "sobol": q_sobol[k],
        "sobol+refine": q_refine[k],
        "ratio": q_refine[k] / q_sobol[k] if q_sobol[k] not in (0, 0.0) else np.nan,
    })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, float_format='%.4f'))

**What to read off:**
- `phi_p` ↓ and `min_dist` ↑ should each shift by ~2-3× — the headline space-filling improvement.
- `coverage` should stay at 1.0 (every level appears at least once for both designs).
- `mir_avg` ↑ in the refined design — the optimizer spreads points in continuous space, slightly hurting per-factor balance. Trade-off worth it for `n ≥ 100` on the comprehensive preset.

## 2. Plug the chosen design into the harness

`DOEGMSBenchmark` reads `config.method` directly — once the config is fixed, no other code changes. Below uses the test document already in the repo and **reuses the pre-trained `ch04_ground_truth` GMS store** so we skip the ~500-epoch training step (the harness detects the existing store on disk).

The demo uses a deliberately small budget (`n_runs=32`, `max_per_category=1`, 2 generators ⇒ ~64 expanded queries) so the full pipeline — load store → expand → graph baseline → optional LLM run → analysis — completes in a couple of minutes. The space-filling improvement was already established at `n=128` in §1; here we exercise the wiring. See §6 for paper-scale settings.

In [ ]:
from importlib.resources import files
N_RUNS_HARNESS = 32       # smaller than the §1 design demo (n=128) so the pipeline runs fast
MAX_PER_CATEGORY = 1      # 1 base question per generator → tiny demo footprint
DEMO_GENERATORS = [       # 2 of 13 default generators → ~64 expanded queries
    "plausibility", "exact_recall",
]

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",   # reuse pre-trained store
    store_name="ch04_ground_truth",
    factor_group=FACTOR_GROUP,
    n_runs=N_RUNS_HARNESS,
    method="sobol+refine",
    seed=SEED,
    max_per_category=MAX_PER_CATEGORY,
    generators=DEMO_GENERATORS,
    ingest_epochs=500,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()    # loads existing store; doesn't retrain
doe_qs = bench.generate_questions()
print(f"Base questions: {len(doe_qs) // config.n_runs}")
print(f"Design runs:    {config.n_runs}")
print(f"Total queries:  {len(doe_qs)}")

## 3. Sanity check: graph baseline must be 100%

Before paying for any LLM calls, verify ground-truth integrity. The graph baseline must answer every DOE-expanded question correctly — if not, the rephrasing has corrupted the ground truth somewhere.

*This step can be slow (graph queries scale with the number of expanded questions). For a quick smoke run, drop `max_per_category` to 2.*

In [ ]:
gt = bench.run_gms_baseline()
print(f"Graph baseline: {gt.total_correct}/{gt.total_questions} ({gt.accuracy:.1%})")
assert gt.accuracy > 0.99, "Graph baseline must be ~100% — ground truth issue otherwise"

## 4. Run the LLM evaluator (optional — costs API calls)

`get_anthropic_api_key()` is the canonical resolver in this codebase: it checks the `ANTHROPIC_API_KEY` env var, falling back to `pass show anthropic/api-key` (Unix `pass(1)` password store). If neither is configured it raises `AnthropicAPIKeyError` — there is no silent mock-LLM fallback in production.

Skip this cell if you don't want to spend API credits. The downstream factor-effect analysis works on any result with `correct` plus the factor columns — substitute a saved result CSV from a prior run.

In [ ]:
from knowlytix.harness.governance.api_key import get_anthropic_api_key, AnthropicAPIKeyError

try:
    api_key = get_anthropic_api_key()
    os.environ["ANTHROPIC_API_KEY"] = api_key  # downstream LLMEvaluator reads from env
    from knowlytix.harness.testing import make_evaluator
    evaluator = make_evaluator("llm", model="claude-sonnet-4-20250514")
    result = bench.run(evaluator=evaluator)
    print(f"LLM accuracy: {result.total_correct}/{result.total_questions} ({result.accuracy:.1%})")
except AnthropicAPIKeyError as e:
    print(f"No Anthropic API key — skipping LLM run. ({e})")
    result = None

## 5. Identify which factors drive failures

`DOEAnalyzer.run_logistic` fits a single-factor logistic regression per factor and reports `G²`, `p_value`, `pseudo_r2`, level rates, odds ratios.

The paper finding: with FDR correction over 21 factors, both designs detect `base_question` plus one or two contextual factors. But the **joint** logistic model — all factors fitted together — fits the refined design substantially better (pseudo-R² 0.825 vs 0.752, ΔAIC = −90.5). That is the operational gain from optimal design: a more informative regression overall, not necessarily more individual stars.

In [ ]:
if result is not None:
    ana = bench.analyze(result)

    deviance = ana.run_logistic(metric="correct")
    print("Top factors by p-value:")
    print(deviance.sort_values("p_value").head(10).to_string(index=False))

    print("\nFailure rates by level (top factor):")
    top = deviance.sort_values("p_value").iloc[0]["factor"]
    print(ana.failure_analysis().query(f"factor == '{top}'").to_string(index=False))
else:
    print("Skipping analysis — no result.")

## 6. Scaling to paper settings

To replicate the paper's case study, change only three knobs:

```python
config = DOEHarnessConfig(
    markdown_path="knowlytix/harness/testing/tests/model_risk_assessment.md",
    stores_dir="optdoe_run/stores",
    store_name="optdoe_paper",
    factor_group="comprehensive",
    n_runs=880,                # paper scale
    method="sobol+refine",
    seed=42,
    max_per_category=44,       # ~44 base questions per the paper
    ingest_epochs=500,
    device="cuda",
)
```

Budget separately for the LLM run (~45 minutes per design at 880 calls against Claude Sonnet) and run a parallel `method="sobol"` baseline for the side-by-side. The optimizer auto-switches from LBFGS (n ≤ 500, float64) to Adam (n > 500, float32, 500 iters, lr = 0.005, p = 50) — no knobs to tune.

## See also

- **User Guide §33.5** — the prose tutorial this notebook accompanies.
- **Jin & Sudjianto, *Conditional Quantile Treatment Effects via Optimal Design* (JASA 2025)** — full methodology, theoretical properties (Fisher information / Schur complement, fill-distance argument), and the 880-run case study.
- **`simulation_power_study.py`** (in the JASA paper's reproducibility supplement) — the simulation that produced the +22pp statistical-power claim under known ground truth.